In [23]:
# 1. Imports
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
import wfdb
import ast
import time
from tqdm import tqdm
from pathlib import Path
import wandb 
wandb.finish()

In [25]:
# 2. Load CSVs
dataset_path = "/users/PLS0150/shreeshtee/PTBXL-Dataset-Thesiswork/physionet.org/files/ptb-xl/1.0.3"

# Dataset files
ptbxl_path = f"{dataset_path}/ptbxl_database.csv"
scp_path = f"{dataset_path}/scp_statements.csv"
waveform_path = dataset_path

# Load CSVs
df = pd.read_csv(ptbxl_path)
scp_df = pd.read_csv(scp_path, index_col=0)

In [29]:
# Parse SCP codes only if they are still strings
df['scp_codes'] = df['scp_codes'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

# Extract SCP code names
df['scp_keys'] = df['scp_codes'].apply(lambda x: list(x.keys()))

# Keep only diagnostic SCP codes
scp_df = scp_df[scp_df['diagnostic'] == 1]

In [30]:
# 3. Select Top 10
target_labels = ['NORM', 'SR', 'AFIB', 'PVC', 'LVH', 'ABQRS', 'IMI', 'ASMI', 'LAFB', 'IRBBB']

In [31]:
# Filter dataset
df['scp_filtered'] = df['scp_keys'].apply(lambda codes: [code for code in codes if code in target_labels])
df = df[df['scp_filtered'].map(len) > 0]

In [32]:
# Binarize
mlb = MultiLabelBinarizer(classes=target_labels)
y = mlb.fit_transform(df['scp_filtered'])

In [33]:
# 4. Split
X_train, X_test, y_train, y_test = train_test_split(df, y, test_size=0.2, random_state=42)

In [34]:
# 5. Dataset
class PTBXL_Dataset(Dataset):
    def __init__(self, df, labels, base_dir, signal_len=5000):
        self.df = df
        self.labels = labels
        self.base_dir = base_dir
        self.signal_len = signal_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = os.path.join(self.base_dir, row['filename_lr'])
        record = wfdb.rdrecord(path)
        signal = record.p_signal.T

        if signal.shape[1] < self.signal_len:
            pad = self.signal_len - signal.shape[1]
            signal = np.pad(signal, ((0, 0), (0, pad)), 'constant')
        else:
            signal = signal[:, :self.signal_len]

        return torch.tensor(signal, dtype=torch.float32), torch.tensor(self.labels[idx], dtype=torch.float32)

train_dataset = PTBXL_Dataset(X_train, y_train, waveform_path)
test_dataset = PTBXL_Dataset(X_test, y_test, waveform_path)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, num_workers=4, pin_memory=True)

In [35]:
# 6. CNN Model
class ECG_CNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(12, 32, kernel_size=7, padding=3),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)
        )
        self.fc = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.cnn(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

In [36]:
# 7. Initialize Weights & Biases
epochs = 15
wandb.init(settings=wandb.Settings(start_method="fork", _disable_stats=False))
wandb.init(
    project="ptbxl-cnn",
    name=f"cnn-multilabel-run-{int(time.time())}",# unique name using timestamp
    settings=wandb.Settings(start_method="fork", _disable_stats=False)
)


wandb.config.update({
    "epochs": epochs,
    "batch_size": 32,
    "learning_rate": 0.001,
    "architecture": "1D CNN Multi-label PTB-XL",
    "dataset": "PTB-XL Top 10 Classes"
})

wandb: WARNING `start_method` is deprecated and will be removed in a future version of wandb. This setting is currently non-functional and safely ignored.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /users/PLS0150/shreeshtee/.netrc.
wandb: Currently logged in as: sdhakal13 (sdhakal13-youngstown-state-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [37]:
# 8. Train
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ECG_CNN(num_classes=len(target_labels)).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(epochs):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    train_loader_tqdm = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")

    for signals, labels in train_loader_tqdm:
        signals, labels = signals.to(device), labels.to(device)
        signals = signals.permute(0, 1, 2)

        optimizer.zero_grad()
        outputs = model(signals)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        preds = (torch.sigmoid(outputs) > 0.5).float()
        acc = (preds == labels).float().mean().item()
        correct += acc
        total += 1

        train_loader_tqdm.set_postfix(loss=loss.item())

    avg_loss = total_loss / len(train_loader)
    avg_acc = correct / total

    print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}, Accuracy: {avg_acc:.4f}")
    wandb.log({"epoch": epoch+1, "loss": avg_loss, "accuracy": avg_acc})


Epoch 1/15: 100%|██████████| 523/523 [00:42<00:00, 12.35it/s, loss=0.303]


Epoch 1, Loss: 0.3726, Accuracy: 0.8504


Epoch 2/15: 100%|██████████| 523/523 [00:15<00:00, 33.81it/s, loss=0.254]


Epoch 2, Loss: 0.3149, Accuracy: 0.8807


Epoch 3/15: 100%|██████████| 523/523 [00:15<00:00, 34.30it/s, loss=0.327]


...

Epoch 13, Loss: 0.2440, Accuracy: 0.9009


Epoch 14/15: 100%|██████████| 523/523 [00:16<00:00, 32.26it/s, loss=0.279]


Epoch 14, Loss: 0.2400, Accuracy: 0.9023


Epoch 15/15: 100%|██████████| 523/523 [00:15<00:00, 33.51it/s, loss=0.287]

Epoch 15, Loss: 0.2353, Accuracy: 0.9040


In [38]:
wandb.finish()

accuracy,▁▅▆▆▆▆▆▇▇▇▇▇███
epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
loss,█▅▄▄▃▃▃▂▂▂▂▂▁▁▁
accuracy,0.90395
epoch,15
loss,0.23533
